# Inicializar directorios

In [1]:
import os
import sys

# ============================================================================
# CONFIGURACIÓN DE DIRECTORIOS
# ============================================================================

# Detectar si estamos en Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Configurar el directorio de trabajo según el entorno
if IN_COLAB:
    project_dir = '/content/TFMDS'
    os.chdir(project_dir)
else:
    # Detectar si estamos en Codespaces o VS Code local
    if os.path.exists('/workspaces/TFMDS'):
        # Entorno Codespaces
        os.chdir('/workspaces/TFMDS')
    else:
        # En VS Code local, nos movemos al directorio raíz del proyecto
        project_dir = r'C:\Users\jmora\Documents\TFMDS'
        os.chdir(project_dir)

# Agregar el directorio del proyecto al path de Python
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Directorio de trabajo:", os.getcwd())
print("Python path incluye proyecto:", os.getcwd() in sys.path)

Directorio de trabajo: /workspaces/TFMDS
Python path incluye proyecto: True


# Importaciones y versiones

In [2]:
# ============================================================================
# IMPORTACIONES BASE Y VERSIÓN
# ============================================================================
import os, sys
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import pytorch_lightning as pl
from neuralforecast import NeuralForecast, __version__ as nf_version
from neuralforecast.models import GRU
from neuralforecast.losses.pytorch import MAE

# Optuna para optimización de hiperparámetros
import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
)

# Importar utilidades desde la carpeta 'lib'
lib_dir = os.path.join(os.getcwd(), 'lib')
if lib_dir not in sys.path:
    sys.path.insert(0, lib_dir)

from lib.dl_utils import (
    preparar_datos_neuralforecast,
    preparar_variables_estaticas,
)
from lib.metricas import calcular_metricas, resumen_metricas
from lib.graficos_dl import (
    grafico_prediccion_diaria_agregada,
    grafico_prediccion_por_cluster,
    grafico_productos_por_cluster,
    dashboard_metricas_dl,
)

# Fijar semilla para reproducibilidad
pl.seed_everything(42, workers=True)

print(f"torch: {torch.__version__}")
print(f"pytorch_lightning: {pl.__version__}")
print(f"neuralforecast: {nf_version}")
print(f"optuna: {optuna.__version__}")

Seed set to 42


torch: 2.9.1+cu128
pytorch_lightning: 2.6.0
neuralforecast: 3.1.2
optuna: 4.6.0


# Lectura de datos

In [3]:
# ============================================================================
# LECTURA DE DATOS
# ============================================================================

print("\n" + "="*100)
print("📂 CARGANDO DATOS PARA DEEP LEARNING")
print("="*100)

df_train_raw = pd.read_csv('datos/df_train_dl.csv', sep=';', parse_dates=['idSecuencia'])
df_test_raw = pd.read_csv('datos/df_test_dl.csv', sep=';', parse_dates=['idSecuencia'])

print(f"\n✅ Datos cargados:")
print(f"   Train: {df_train_raw.shape}")
print(f"   Test:  {df_test_raw.shape}")


📂 CARGANDO DATOS PARA DEEP LEARNING

✅ Datos cargados:
   Train: (625800, 27)
   Test:  (27714, 27)

✅ Datos cargados:
   Train: (625800, 27)
   Test:  (27714, 27)


# Preparación de datos para NeuralForecast

In [4]:
# ============================================================================
# PREPARACIÓN DE DATOS PARA NEURALFORECAST
# ============================================================================

print("\n" + "="*100)
print("🔧 PREPARANDO DATOS PARA NEURALFORECAST")
print("="*100)

# Convertir al formato NeuralForecast (unique_id, ds, y)
df_train_nf, df_test_nf = preparar_datos_neuralforecast(
    df_train_raw,
    df_test_raw,
    col_fecha='idSecuencia',
    col_producto='producto',
    col_target='udsVenta'
)

print(f"\n✅ Datos convertidos a formato NeuralForecast")
print(f"   Train NF: {df_train_nf.shape}")
print(f"   Test NF:  {df_test_nf.shape}")


🔧 PREPARANDO DATOS PARA NEURALFORECAST

✅ Datos convertidos a formato NeuralForecast
   Train NF: (625800, 27)
   Test NF:  (27714, 27)

✅ Datos convertidos a formato NeuralForecast
   Train NF: (625800, 27)
   Test NF:  (27714, 27)


## Preparar variables estáticas

In [5]:
# ============================================================================
# PREPARAR VARIABLES ESTÁTICAS
# ============================================================================

print("\n" + "="*100)
print("🔧 PREPARANDO VARIABLES ESTÁTICAS")
print("="*100)

# Definir columnas estáticas (que no varían en el tiempo)
stat_exog_list = ['Cluster_0', 'Cluster_1', 'Cluster_2', 'Cluster_3']

# Separar variables estáticas en DataFrame aparte
df_train_nf, df_test_nf, static_df = preparar_variables_estaticas(
    df_train_nf,
    df_test_nf,
    stat_exog_list
)



🔧 PREPARANDO VARIABLES ESTÁTICAS

✅ Variables estáticas extraídas:
   Columnas estáticas: ['Cluster_0', 'Cluster_1', 'Cluster_2', 'Cluster_3']
   Shape static_df: (894, 5)
   Productos únicos: 894

📊 Datasets temporales actualizados:
   Train: (625800, 23) (eliminadas 4 columnas)
   Test:  (27714, 23) (eliminadas 4 columnas)


## Limpieza de features históricas

In [6]:
# =============================================================================
# LIMPIEZA DE FEATURES HISTÓRICAS (evitar NaN al inicio del test)
# =============================================================================

print("\n" + "="*100)
print("🧹 LIMPIANDO FEATURES HISTÓRICAS")
print("="*100)

# Definir la lista de features históricas que usa el modelo
hist_exog_list = [
    'lag_ventas_1', 'lag_ventas_2', 'lag_ventas_3', 'lag_ventas_4',
    'lag_ventas_5', 'lag_ventas_6', 'lag_ventas_7', 'media_mes_anterior',
    'EWMA_corto', 'EWMA_largo', 'Tendencia_EWMA'
]

cols_ok = [c for c in hist_exog_list if c in df_train_nf.columns and c in df_test_nf.columns]

if cols_ok:
    na_train_before = int(df_train_nf[cols_ok].isna().sum().sum())
    na_test_before = int(df_test_nf[cols_ok].isna().sum().sum())

    # Forward fill por serie (unique_id) y rellenar remanentes al inicio con 0
    df_train_nf[cols_ok] = (
        df_train_nf.groupby('unique_id', observed=True)[cols_ok]
        .ffill()
        .fillna(0)
    )
    df_test_nf[cols_ok] = (
        df_test_nf.groupby('unique_id', observed=True)[cols_ok]
        .ffill()
        .fillna(0)
    )

    # Asegurar tipos numéricos
    for df_tmp in (df_train_nf, df_test_nf):
        for c in cols_ok:
            df_tmp[c] = pd.to_numeric(df_tmp[c], errors='coerce').fillna(0)

    na_train_after = int(df_train_nf[cols_ok].isna().sum().sum())
    na_test_after = int(df_test_nf[cols_ok].isna().sum().sum())

    print(f"✅ NaNs en train: {na_train_before} → {na_train_after}")
    print(f"✅ NaNs en test:  {na_test_before} → {na_test_after}")


🧹 LIMPIANDO FEATURES HISTÓRICAS
✅ NaNs en train: 4 → 0
✅ NaNs en test:  0 → 0
✅ NaNs en train: 4 → 0
✅ NaNs en test:  0 → 0


# División de datos para validación

Para optimizar hiperparámetros, necesitamos un conjunto de validación.
Dividiremos el conjunto de entrenamiento en train/validation usando los últimos 30 días como validación.

In [7]:
# ============================================================================
# DIVISIÓN TRAIN/VALIDATION
# ============================================================================

print("\n" + "="*100)
print("📊 DIVISIÓN TRAIN/VALIDATION PARA OPTUNA")
print("="*100)

# Horizonte de predicción y validación
HORIZON = 30
VAL_SIZE = 30  # Últimos 30 días del train para validación

# Encontrar la fecha de corte para validación
max_date_train = df_train_nf['ds'].max()
val_cutoff = max_date_train - pd.Timedelta(days=VAL_SIZE)

# Dividir en train y validation
df_train_opt = df_train_nf[df_train_nf['ds'] <= val_cutoff].copy()
df_val_opt = df_train_nf[df_train_nf['ds'] > val_cutoff].copy()

print(f"\n✅ División completada:")
print(f"   Train para optimización: {df_train_opt.shape}")
print(f"   Validation: {df_val_opt.shape}")
print(f"   Test final: {df_test_nf.shape}")
print(f"\n   Fecha de corte validación: {val_cutoff}")
print(f"   Rango train: {df_train_opt['ds'].min()} a {df_train_opt['ds'].max()}")
print(f"   Rango val:   {df_val_opt['ds'].min()} a {df_val_opt['ds'].max()}")


📊 DIVISIÓN TRAIN/VALIDATION PARA OPTUNA

✅ División completada:
   Train para optimización: (598980, 23)
   Validation: (26820, 23)
   Test final: (27714, 23)

   Fecha de corte validación: 2024-09-05 00:00:00
   Rango train: 2022-11-06 00:00:00 a 2024-09-05 00:00:00
   Rango val:   2024-09-06 00:00:00 a 2024-10-05 00:00:00

✅ División completada:
   Train para optimización: (598980, 23)
   Validation: (26820, 23)
   Test final: (27714, 23)

   Fecha de corte validación: 2024-09-05 00:00:00
   Rango train: 2022-11-06 00:00:00 a 2024-09-05 00:00:00
   Rango val:   2024-09-06 00:00:00 a 2024-10-05 00:00:00


# Configuración de variables exógenas

Definimos las listas de variables que usará el modelo.

In [8]:
# ============================================================================
# CONFIGURACIÓN DE VARIABLES EXÓGENAS
# ============================================================================

# Variables futuras (conocidas en el horizonte de predicción)
futr_exog_list = [
    'bolOpen', 'bolHoliday', 'bolPromocion',
    'dia_semana_sin', 'dia_semana_cos',
    'mes_sin', 'mes_cos',
    'trimestre_sin', 'trimestre_cos'
]

# Variables históricas (solo disponibles en el pasado)
hist_exog_list = [
    'lag_ventas_1', 'lag_ventas_2', 'lag_ventas_3', 'lag_ventas_4',
    'lag_ventas_5', 'lag_ventas_6', 'lag_ventas_7', 'media_mes_anterior',
    'EWMA_corto', 'EWMA_largo', 'Tendencia_EWMA'
]

print(f"Variables futuras: {len(futr_exog_list)}")
print(f"Variables históricas: {len(hist_exog_list)}")
print(f"Variables estáticas: {len(stat_exog_list)}")

Variables futuras: 9
Variables históricas: 11
Variables estáticas: 4


# Función objetivo para Optuna

Esta función define qué hiperparámetros optimizar y cómo evaluar cada configuración.

In [9]:
# ============================================================================
# FUNCIÓN OBJETIVO PARA OPTUNA - GRU
# ============================================================================

def objective(trial):
    """
    Función objetivo para Optuna con modelo GRU.
    Retorna el MAE en el conjunto de validación.
    """
    
    # Hiperparámetros a optimizar
    params = {
        'input_size': trial.suggest_int('input_size', 30, 90, step=15),
        'encoder_hidden_size': trial.suggest_categorical('encoder_hidden_size', [64, 128, 256]),
        'encoder_n_layers': trial.suggest_int('encoder_n_layers', 1, 3),
        'decoder_hidden_size': trial.suggest_categorical('decoder_hidden_size', [64, 128, 256]),
        'decoder_layers': trial.suggest_int('decoder_layers', 1, 3),
        'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [16, 32, 64]),
        'scaler_type': trial.suggest_categorical('scaler_type', ['standard', 'robust', 'minmax']),
        'max_steps': trial.suggest_int('max_steps', 1000, 2000, step=200),
    }
    
    try:
        # Filtrar static_df a los unique_id presentes en train/val y asegurar columnas numéricas
        unique_ids_train = df_train_opt['unique_id'].unique()
        static_df_opt = static_df[static_df['unique_id'].isin(unique_ids_train)].copy()
        # Comprobar columnas esperadas
        missing_stat = [c for c in stat_exog_list if c not in static_df_opt.columns]
        if missing_stat:
            raise ValueError(f"Faltan columnas estáticas en static_df: {missing_stat}")
        # Tipos numéricos por seguridad
        for c in stat_exog_list:
            static_df_opt[c] = pd.to_numeric(static_df_opt[c], errors='coerce').fillna(0)

        # Crear modelo GRU con los parámetros sugeridos
        modelo = GRU(
            h=HORIZON,
            input_size=params['input_size'],
            loss=MAE(),
            max_steps=params['max_steps'],
            encoder_hidden_size=params['encoder_hidden_size'],
            encoder_n_layers=params['encoder_n_layers'],
            decoder_hidden_size=params['decoder_hidden_size'],
            decoder_layers=params['decoder_layers'],
            learning_rate=params['learning_rate'],
            scaler_type=params['scaler_type'],
            batch_size=params['batch_size'],
            random_seed=42,
            futr_exog_list=futr_exog_list,
            hist_exog_list=hist_exog_list,
            stat_exog_list=stat_exog_list,
            enable_progress_bar=False,
        )
        
        # Crear NeuralForecast y entrenar
        nf = NeuralForecast(models=[modelo], freq='D')
        nf.fit(df=df_train_opt, static_df=static_df_opt)
        
        # Predecir en validación
        y_hat = nf.predict(futr_df=df_val_opt)
        
        # Preparar predicciones
        if y_hat.index.name == 'unique_id':
            y_hat = y_hat.reset_index()
        
        if 'GRU' in y_hat.columns:
            y_hat = y_hat.rename(columns={'GRU': 'prediccion'})
        else:
            # fallback: primera columna de predicción
            pred_cols = [c for c in y_hat.columns if c not in ['unique_id', 'ds']]
            if not pred_cols:
                raise ValueError("No se encontró columna de predicción en y_hat.")
            y_hat = y_hat.rename(columns={pred_cols[0]: 'prediccion'})
        
        # Merge con valores reales
        df_val_pred = df_val_opt[['unique_id', 'ds', 'y']].merge(
            y_hat[['unique_id', 'ds', 'prediccion']],
            on=['unique_id', 'ds'],
            how='left'
        )
        
        # Clip de valores negativos
        df_val_pred['prediccion'] = df_val_pred['prediccion'].clip(lower=0)
        
        # Calcular MAE en validación
        df_valid = df_val_pred.dropna(subset=['prediccion', 'y'])
        mae = np.abs(df_valid['prediccion'] - df_valid['y']).mean()
        
        return mae
        
    except Exception as e:
        print(f"\n⚠️ Error en trial {trial.number}: {str(e)}")
        # Retornar un valor alto si falla
        return 1e6

print("\n✅ Función objetivo para GRU configurada")


✅ Función objetivo para GRU configurada


# Ejecutar optimización con Optuna

⚠️ **NOTA:** Este proceso puede tardar bastante tiempo dependiendo de `n_trials`.
- Para pruebas rápidas: `n_trials=10-20` (~30-60 min)
- Para búsqueda exhaustiva: `n_trials=50-100` (varias horas)

In [10]:
# ============================================================================
# OPTIMIZACIÓN CON OPTUNA - GRU
# ============================================================================

print("\n" + "="*100)
print("🔍 INICIANDO BÚSQUEDA DE HIPERPARÁMETROS CON OPTUNA - GRU")
print("="*100)

# Configuración del estudio
N_TRIALS = 20  # Número de configuraciones a probar (ajustar según tiempo disponible)

# Crear estudio Optuna (minimizar MAE)
study = optuna.create_study(
    direction='minimize',
    study_name='gru_hyperparameter_optimization',
    sampler=optuna.samplers.TPESampler(seed=42),  # Tree-structured Parzen Estimator
)

print(f"\n⏳ Ejecutando {N_TRIALS} trials...")
print("   Esto puede tardar bastante tiempo.\n")

start_time = time.perf_counter()

# Ejecutar optimización (sin barra de progreso para evitar conflictos en notebooks)
study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=False,
    n_jobs=1,  # Secuencial en CPU
)

elapsed = time.perf_counter() - start_time

print(f"\n✅ Optimización completada en {elapsed:.2f} segundos ({elapsed/60:.2f} minutos)")
print(f"\n🏆 MEJORES HIPERPARÁMETROS ENCONTRADOS:")
print("="*100)
for key, value in study.best_params.items():
    print(f"   {key}: {value}")
print(f"\n   Mejor MAE en validación: {study.best_value:.4f}")
print("="*100)

[I 2025-11-29 10:55:54,138] A new study created in memory with name: gru_hyperparameter_optimization
Seed set to 42
Seed set to 42



🔍 INICIANDO BÚSQUEDA DE HIPERPARÁMETROS CON OPTUNA - GRU

⏳ Ejecutando 20 trials...
   Esto puede tardar bastante tiempo.



GPU available: False, used: False
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | loss         | MAE           | 0      | train | 0    
1 | padder_train | ConstantPad1d | 0      | train | 0    
2 | scaler       | TemporalNorm  | 0      | train | 0    
3 | hist_encoder | GRU           | 17.5 K | train | 0    
4 | mlp_decoder  | MLP           | 19.2 K | train | 0    
---------------------------------------------------------------
36.7 K    Trainable params
0         Non-trainable params
36.7 K    Total params
0.147     Total estimated model params size (MB)
10        Modules in train mode
0         Modules in eval mode
0         Total Flops

  | Name         | Type          | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | loss         | MAE           | 0      | train | 0    
1 | padder_

SIGTERMException: 

# Visualización de la optimización

In [ ]:
# ============================================================================
# VISUALIZACIÓN DE RESULTADOS DE OPTUNA
# ============================================================================

print("\n" + "="*100)
print("📊 VISUALIZACIONES DE LA OPTIMIZACIÓN")
print("="*100)

# 1. Historia de optimización
fig1 = plot_optimization_history(study)
fig1.update_layout(title='Historia de Optimización GRU - MAE en Validación')
fig1.show()

# 2. Importancia de parámetros
fig2 = plot_param_importances(study)
fig2.update_layout(title='Importancia de Hiperparámetros - GRU')
fig2.show()

# 3. Coordenadas paralelas
fig3 = plot_parallel_coordinate(study)
fig3.update_layout(title='Coordenadas Paralelas de Hiperparámetros - GRU')
fig3.show()

print("\n✅ Visualizaciones generadas")

# Entrenamiento del modelo final con mejores hiperparámetros

Ahora entrenamos el modelo con los mejores hiperparámetros encontrados,
usando el conjunto completo de entrenamiento.

In [ ]:
# ============================================================================
# ENTRENAMIENTO FINAL CON MEJORES HIPERPARÁMETROS - GRU
# ============================================================================

print("\n" + "="*100)
print("🚀 ENTRENAMIENTO MODELO FINAL GRU CON MEJORES HIPERPARÁMETROS")
print("="*100)

# Obtener mejores parámetros
best_params = study.best_params

# Crear modelo GRU optimizado
modelo_gru_optimizado = GRU(
    h=HORIZON,
    input_size=best_params['input_size'],
    loss=MAE(),
    max_steps=best_params['max_steps'],
    encoder_hidden_size=best_params['encoder_hidden_size'],
    encoder_n_layers=best_params['encoder_n_layers'],
    decoder_hidden_size=best_params['decoder_hidden_size'],
    decoder_layers=best_params['decoder_layers'],
    learning_rate=best_params['learning_rate'],
    scaler_type=best_params['scaler_type'],
    batch_size=best_params['batch_size'],
    random_seed=42,
    futr_exog_list=futr_exog_list,
    hist_exog_list=hist_exog_list,
    stat_exog_list=stat_exog_list,
    enable_progress_bar=True,
)

# Crear NeuralForecast
nf_final = NeuralForecast(
    models=[modelo_gru_optimizado],
    freq='D'
)

# Entrenar con TODO el conjunto de entrenamiento
print("\n⏳ Entrenando modelo final con mejores hiperparámetros...")
start_time = time.perf_counter()

nf_final.fit(df=df_train_nf, static_df=static_df)

elapsed = time.perf_counter() - start_time

print(f"\n✅ Entrenamiento completado en {elapsed:.2f} segundos ({elapsed/60:.2f} minutos)")

# Predicción sobre el conjunto de test

In [ ]:
# ============================================================================
# PREDICCIÓN EN TEST
# ============================================================================

print("\n" + "="*100)
print("🔮 GENERANDO PREDICCIONES EN TEST")
print("="*100)

print("\n⏳ Generando predicciones en test set...")
start_time = time.perf_counter()

y_hat = nf_final.predict(futr_df=df_test_nf)

elapsed = time.perf_counter() - start_time

print(f"\n✅ Predicciones generadas en {elapsed:.2f} segundos")
print(f"   Shape predicciones: {y_hat.shape}")

# Preparar predicciones para evaluación

In [ ]:
# ============================================================================
# PREPARAR PREDICCIONES
# ============================================================================

print("\n" + "="*100)
print("🔧 PREPARANDO PREDICCIONES PARA MÉTRICAS")
print("="*100)

# Normalizar índice si es necesario
if y_hat.index.name == 'unique_id':
    y_hat = y_hat.reset_index()

# Renombrar columna de predicción
if 'GRU' in y_hat.columns:
    y_hat = y_hat.rename(columns={'GRU': 'prediccion'})

# Merge con valores reales
df_test_pred = df_test_nf[['unique_id', 'ds', 'y']].merge(
    y_hat[['unique_id', 'ds', 'prediccion']],
    on=['unique_id', 'ds'],
    how='left'
)

# Columnas compatibles con funciones existentes
df_test_pred['idSecuencia'] = df_test_pred['ds']
df_test_pred['producto'] = df_test_pred['unique_id']
df_test_pred['udsVenta'] = df_test_pred['y']

# Clip de valores negativos a 0
df_test_pred['prediccion'] = df_test_pred['prediccion'].clip(lower=0)

# Calcular errores
df_test_pred['error'] = df_test_pred['prediccion'] - df_test_pred['udsVenta']
df_test_pred['error_abs'] = np.abs(df_test_pred['error'])

print(f"\n✅ Dataset de predicciones listo:")
print(f"   Shape: {df_test_pred.shape}")
print(f"   Predicciones no nulas: {df_test_pred['prediccion'].notna().sum()}")
print(f"\n📊 Estadísticas del error:")
print(f"   Error medio: {df_test_pred['error'].mean():.2f} unidades")
print(f"   Error abs medio: {df_test_pred['error_abs'].mean():.2f} unidades")

# Cálculo de métricas en test

In [ ]:
# ============================================================================
# CÁLCULO DE MÉTRICAS EN TEST
# ============================================================================

print("\n" + "="*100)
print("📊 CÁLCULO DE MÉTRICAS EN TEST")
print("="*100)

# Filtrar valores válidos
df_valid = df_test_pred.dropna(subset=['prediccion', 'udsVenta'])

# Calcular métricas
metricas_gru_optuna = calcular_metricas(
    y=df_valid['udsVenta'],
    y_pred=df_valid['prediccion'],
    name='GRU_Optuna'
)

# Mostrar resumen
resumen_metricas([metricas_gru_optuna])

todas_metricas = [metricas_gru_optuna]

# Reconstruir columna Cluster

In [ ]:
# ============================================================================
# RECONSTRUIR COLUMNA CLUSTER
# ============================================================================

if 'Cluster' not in df_test_pred.columns:
    cluster_cols = [c for c in static_df.columns if c.startswith('Cluster_')]
    
    if cluster_cols:
        cluster_cols = sorted(cluster_cols, key=lambda x: int(x.split('_')[1]))
        static_map = static_df[['unique_id'] + cluster_cols].copy()
        cluster_idx = static_map[cluster_cols].to_numpy().argmax(axis=1)
        static_map['Cluster'] = cluster_idx
        
        df_test_pred = df_test_pred.merge(
            static_map[['unique_id', 'Cluster']],
            on='unique_id',
            how='left'
        )
        
        print(f"\n✅ 'Cluster' reconstruido desde static_df")
        print("   Distribución:")
        print(df_test_pred['Cluster'].value_counts(dropna=False).sort_index())

# Visualizaciones

## 1. Predicción diaria agregada

In [ ]:
grafico_prediccion_diaria_agregada(
    df=df_test_pred,
    col_fecha='idSecuencia',
    col_real='udsVenta',
    col_pred='prediccion',
    titulo='GRU Optuna - Ventas Diarias Agregadas (Todos los Productos)',
    figsize=(14, 5)
)

## 2. Predicciones por Cluster

In [ ]:
grafico_prediccion_por_cluster(
    df=df_test_pred,
    col_cluster='Cluster',
    col_fecha='idSecuencia',
    col_real='udsVenta',
    col_pred='prediccion',
    figsize=(16, 10)
)

## 3. Top 2 productos por Cluster

In [ ]:
grafico_productos_por_cluster(
    df=df_test_pred,
    col_cluster='Cluster',
    col_producto='producto',
    col_fecha='idSecuencia',
    col_real='udsVenta',
    col_pred='prediccion',
    n_productos_por_cluster=2,
    figsize=(18, 12)
)

## 4. Dashboard de métricas

In [ ]:
# Preparar diccionario de métricas
metricas_dict = {}
for m in todas_metricas:
    nombre_algoritmo = m['Algoritmo']
    metricas_dict[nombre_algoritmo] = {k: v for k, v in m.items() if k != 'Algoritmo'}

dashboard_metricas_dl(
    metricas_dict=metricas_dict,
    titulo='Dashboard de Métricas - GRU Optimizado con Optuna',
    figsize=(16, 10)
)

# Guardar resultados

Guardamos tanto las métricas como los mejores hiperparámetros encontrados.

In [ ]:
# ============================================================================
# GUARDAR RESULTADOS - GRU
# ============================================================================

print("\n" + "="*100)
print("💾 GUARDANDO RESULTADOS")
print("="*100)

# 1. Guardar métricas
df_resultados = pd.DataFrame(todas_metricas)
output_path_metricas = 'datos/resultados_metricas_gru_optuna.csv'
df_resultados.to_csv(output_path_metricas, index=False)
print(f"\n✅ Métricas guardadas en: {output_path_metricas}")

# 2. Guardar mejores hiperparámetros
df_best_params = pd.DataFrame([study.best_params])
df_best_params['best_mae_validation'] = study.best_value
output_path_params = 'datos/mejores_hiperparametros_gru_optuna.csv'
df_best_params.to_csv(output_path_params, index=False)
print(f"✅ Mejores hiperparámetros guardados en: {output_path_params}")

# 3. Guardar historial completo de trials
df_trials = study.trials_dataframe()
output_path_trials = 'datos/historial_trials_gru_optuna.csv'
df_trials.to_csv(output_path_trials, index=False)
print(f"✅ Historial de trials guardado en: {output_path_trials}")

print("\n📊 Resumen de métricas en test:")
print(df_resultados.to_string(index=False))

print("\n🏆 Mejores hiperparámetros:")
print(df_best_params.to_string(index=False))
print("="*100)

# Resumen final

## Comparación con modelo base

Para comparar este modelo optimizado con el modelo base (sin Optuna),
ejecuta el cuaderno `06_GRU.ipynb` y compara las métricas.

In [ ]:
# ============================================================================
# RESUMEN FINAL
# ============================================================================

print("\n" + "="*100)
print("📋 RESUMEN FINAL DE LA OPTIMIZACIÓN - GRU")
print("="*100)

print(f"\n🔍 Configuración de búsqueda:")
print(f"   Trials ejecutados: {len(study.trials)}")
print(f"   Tiempo total: {elapsed/60:.2f} minutos")

print(f"\n🏆 Mejor configuración encontrada:")
for key, value in study.best_params.items():
    print(f"   {key}: {value}")

print(f"\n📊 Rendimiento:")
print(f"   MAE en validación: {study.best_value:.4f}")
print(f"   MAE en test: {metricas_gru_optuna['MAE']:.4f}")
print(f"   RMSE en test: {metricas_gru_optuna['RMSE']:.4f}")

print("\n💡 Próximos pasos:")
print("   1. Comparar con el modelo GRU base (06_GRU.ipynb)")
print("   2. Comparar con los modelos LSTM (base y optimizado)")
print("   3. Si mejora significativa, usar estos hiperparámetros como base")
print("   4. Analizar la importancia de hiperparámetros en las visualizaciones")
print("="*100)